# Using Different bacpipe Functions

This notebook is a collection of short, mostly independent code blocks that
demonstrate the individual building blocks of `bacpipe`, complementing the
end-to-end workflow in `simple_use_cases.ipynb`. Every object that `bacpipe`
exposes at the top level (`bacpipe.__all__`) is shown here in a small,
self-contained snippet, so you can copy any cell and adapt it to your own use
case.

Most cells only depend on the **Setup & Configuration** cell below. A few
evaluation cells additionally reuse the embeddings computed in the
*Generating Embeddings* section; those dependencies are called out in the
section text.


Make sure to reset your config and settings files before running this notebook, 
as they may contain settings from previous runs that could prevent this notebook 
from finding the correct paths.

---

**Contents**

1. [Setup & Configuration](#1.-Setup-&-Configuration)
2. [Constants](#2.-Constants)
3. [Folder Structure & Paths](#3.-Folder-Structure-&-Paths)
4. [Model Utilities](#4.-Model-Utilities)
5. [Audio Handling](#5.-Audio-Handling)
6. [Embeddings from Audio](#6.-Embeddings-from-Audio)
7. [Generating Embeddings](#7.-Generating-Embeddings)
8. [Custom Metadata Labels](#8.-Custom-Metadata-Labels)
9. [Model-Specific Evaluation](#9.-Model-Specific-Evaluation)
10. [Probing](#10.-Probing)
11. [Clustering](#11.-Clustering)
12. [Benchmarking](#12.-Benchmarking)
13. [Cross-Model Evaluation](#13.-Cross-Model-Evaluation)
14. [Dashboard & play](#14.-Dashboard-&-play)


---
## 1. Setup & Configuration

Set the working directory to the repo root, import the display utilities and
point bacpipe at a dedicated results folder so this tutorial does not
interfere with the others.

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display, Image
from PIL import Image as PILImage
import os
import shutil
from pathlib import Path

import importlib.resources as pkg_resources
import numpy as np
import pandas as pd

os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')

# load the specific package
import bacpipe

# Store the results of this tutorial in a dedicated subfolder so they do not
# interfere with the other tutorials.
bacpipe.settings.main_results_dir = str(
    Path(bacpipe.settings.main_results_dir) / 'using_different_bacpipe_functions'
)

# Keep the tutorial reproducible on any machine.
bacpipe.settings.device = 'cpu'

# !WARNING! the following code deletes the folder where the results of this
# tutorial are stored, to make sure every run starts from a clean folder.
# Comment this out if you have important data in this folder.
shutil.rmtree(bacpipe.settings.main_results_dir, ignore_errors=True)

print('Results will be stored in:', bacpipe.settings.main_results_dir)


def show_half(path):
    # Display a saved plot at half its original size.
    with PILImage.open(path) as img:
        w, _ = img.size
    display(Image(filename=str(path), width=w // 2))


---
## 2. Constants

`bacpipe` re-exports a few module-level constants that describe which models
are supported and which ones need a checkpoint to be downloaded.

In [ ]:
print('supported_models ({}):'.format(len(bacpipe.supported_models)))
print(bacpipe.supported_models)

print()
print('models_needing_checkpoint:')
print(bacpipe.models_needing_checkpoint)

print()
print('TF_MODELS:')
print(bacpipe.TF_MODELS)

print()
print('EMBEDDING_DIMENSIONS (first 6):')
print(dict(list(bacpipe.EMBEDDING_DIMENSIONS.items())[:6]))

print()
print('NEEDS_CHECKPOINT:')
print(bacpipe.NEEDS_CHECKPOINT)


---
## 3. Folder Structure & Paths

Every artifact bacpipe produces (embeddings, labels, probing and clustering
results, plots, ...) is stored in a deterministic folder structure. The layout
is exposed through a lightweight `get_paths` function returned by
`make_set_paths_func`.

In [ ]:
get_paths = bacpipe.make_set_paths_func(
    'bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
)

paths = get_paths('birdnet')

for attr in ['dataset_path', 'main_embeds_path', 'labels_path',
             'probe_path', 'clust_path', 'preds_path', 'plot_path']:
    print(f'{attr:<20} {getattr(paths, attr)}')


In [ ]:
# Check whether the full evaluation (embeddings, dimensionality reduction,
# probing and clustering) already exists for a given model/dataset
# combination. This is False on a fresh run and True once the sections below
# have completed.
already_exists = bacpipe.evaluation_with_settings_already_exists(
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap',
    models=['birdnet', 'insect459'],
)
print('Evaluation already exists:', already_exists)


---
## 4. Model Utilities

Before any processing, make sure the model checkpoints are available locally
and confirm that the requested model names are supported.

In [ ]:
model_base_path = bacpipe.ensure_models_exist(
    bacpipe.settings.model_base_path,
    model_names=['birdnet', 'insect459'],
)
print('Model checkpoints are stored in:', model_base_path)

for model in ['birdnet', 'insect459']:
    print(f'{model} -> {bacpipe.confirm_model_name(model)}')

# get_model_names() either resolves a list of model names against the
# supported models or, with already_computed=True, infers them from the
# embeddings that already exist on disk.
model_names = bacpipe.get_model_names(
    models=['birdnet', 'insect459'],
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
    embed_parent_dir=bacpipe.settings.embed_parent_dir,
)
print('Resolved model names:', model_names)


---
## 5. Audio Handling

`AudioHandler` loads and windows audio the way the selected model needs it.
It only needs the sample rate and the segment length of a model, so passing the
*name* of a supported model is enough -- the model itself (and with it its
checkpoint) is only loaded once the model specific preprocessing is applied in
`prepare_audio()`. A model object (`Embedder(...).model`) can be passed just as
well.

There are two ways to load audio:

1. **whole files**, via `load_and_resample()` + `window_audio()`, and
2. **only the annotated segments**, via `only_load_annotated_segments()`.

In [ ]:
from bacpipe import get_audio_files, AudioHandler

# (1) Whole-file loading: resample to the model sample rate, then window the
# audio into the fixed-length windows the model expects. Passing the model
# name is enough here, no checkpoint is loaded for this.
aud = AudioHandler(model='birdnet', audio_dir='bacpipe/tests/test_data')
print('birdnet expects {} Hz and {} samples per window.'.format(
    aud.model.sr, aud.model.segment_length))

files = get_audio_files('bacpipe/tests/test_data')

all_frames = []
for audio_file in files:
    audio, sr = aud.load_and_resample(audio_file)
    frames = aud.window_audio(audio)
    all_frames.extend(frames)
all_frames = np.stack(all_frames)
print('Windowed audio shape (n_windows, window_length):', all_frames.shape)


In [ ]:
# (2) Annotated-segment loading: extract just the time spans covered by the
# annotations file instead of a regular time grid.
aud_annotated = AudioHandler(
    model='birdnet',
    audio_dir='bacpipe/tests/test_data',
    only_embed_annotations=True,
)

all_annotated_frames = []
for audio_file in files:
    frames = aud_annotated.only_load_annotated_segments(audio_file)
    all_annotated_frames.extend(frames)
all_annotated_frames = np.stack(all_annotated_frames)
print('Annotated segments shape (n_segments, segment_length):',
      all_annotated_frames.shape)


### Deviating from the model defaults

The sample rate and the segment length that were read from the model can be
changed at any time by setting `aud.model.sr` and `aud.model.segment_length`.
The audio is then loaded and windowed exactly the way you asked for, and the
values are kept when the model itself is loaded later on, so the segments are
preprocessed the way they were loaded.

In [ ]:
# Load the audio at 32 kHz and window it into 3 second segments instead of
# the 48 kHz / 3 second windows birdnet uses by default.
aud.model.sr = 32_000
aud.model.segment_length = 3 * aud.model.sr

all_frames = []
for audio_file in files:
    audio, sr = aud.load_and_resample(audio_file)
    frames = aud.window_audio(audio)
    all_frames.extend(frames)
all_frames = np.stack(all_frames)
print('3 s windows at 32 kHz (n_windows, window_length):', all_frames.shape)


### Passing your own annotations to the audio handler

`only_load_annotated_segments()` reads `annotations.csv` from the audio
directory by default. Your own annotations can be passed as a dataframe
instead: only the segments it annotates are returned, everything else in the
recordings is ignored. If the dataframe has an `audiofilename` column it is
filtered down to the current file, so the whole table can be passed for every
file, and time windows that several rows share (e.g. two species in one
window) are loaded once.

In [ ]:
import pandas as pd

annots = pd.read_csv('bacpipe/tests/test_data/annotations.csv')
# only keep the annotations of the first 10 seconds of every recording
annots = annots[annots.start < 10]
print('Annotations that are kept:', len(annots))

all_annotated_frames = []
for audio_file in files:
    frames = aud_annotated.only_load_annotated_segments(
        audio_file, annotations_df=annots
    )
    all_annotated_frames.extend(frames)
all_annotated_frames = np.stack(all_annotated_frames)
print('Segments annotated in the dataframe (n_segments, segment_length):',
      all_annotated_frames.shape)


---
## 6. Embeddings from Audio

`Embedder` accepts just a model name and computes embeddings directly from an
audio file through `get_embeddings_from_model()`.

In [ ]:
embed_obj = bacpipe.Embedder(model_name='birdnet')
audio_files = bacpipe.get_audio_files('bacpipe/tests/test_data')

all_embeds = []
for audio_file in audio_files:
    embeds = embed_obj.get_embeddings_from_model(audio_file)
    all_embeds.extend(embeds)
all_embeds = np.stack(all_embeds)
print('Embeddings shape (n_segments, n_dimensions):', all_embeds.shape)


---
## 7. Generating Embeddings

The same machinery is exposed at three levels: `generate_embeddings` (one
model, embeddings only), `run_pipeline_for_single_model` (one model, full
pipeline incl. classifier and dimensionality reduction) and
`run_pipeline_for_models` (several models, returns a dictionary of loaders).

`Loader` objects are returned by all three and give access to the embeddings
and metadata.

In [ ]:
# generate_embeddings(): single-model embedding generation. With
# check_if_already_processed=True (the default) it reuses existing results.
loader = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data',
    check_if_already_processed=True,
)

# Loader.embeddings() returns a dict mapping file stems to numpy arrays ...
embeds_dict = loader.embeddings()
print('Embeddings per file:', {k: v.shape for k, v in embeds_dict.items()})
# ... or a single numpy array of shape (n_segments, n_dimensions).
print('Embeddings array:', loader.embeddings(return_type='array').shape)


In [ ]:
# run_pipeline_for_single_model(): full pipeline for one model. Set
# check_if_already_processed=False to force recomputation.
loader = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='None',
    check_if_already_processed=False,
)
print('Embeddings array shape:', loader.embeddings(return_type='array').shape)


In [ ]:
# run_pipeline_for_models(): several models at once. Passing
# dim_reduction_model='umap' additionally computes a 2D projection of each
# embedding space, which is used for the cross-model comparison in section 13.
loader_dict = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'insect459'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap',
)

for model, loader in loader_dict.items():
    print(model, '->', loader.embeddings(return_type='array').shape)


---
## 8. Custom Metadata Labels

The high-level `metadata_labels` helper creates the default label set, while
`MetadataLabelMaker` gives full control over which labels are generated.
`ground_truth_by_model` builds the ground-truth dataframe (see section 10) and
`get_dt_filename` extracts a recording timestamp from a filename.

In [ ]:
# MetadataLabelMaker: full control over which metadata labels are created.
label_maker = bacpipe.MetadataLabelMaker(
    paths=get_paths('birdnet'),
    model='birdnet',
    metadata_label_keys=['audio_file_name', 'time_of_day'],
)
label_maker.generate()

metadata_label_dict = label_maker.metadata_label_dict
print('Generated label keys:', list(metadata_label_dict.keys()))
display(pd.DataFrame(metadata_label_dict).head())


In [ ]:
# metadata_labels(): the high-level equivalent, builds the default label set
# for a model in one call (requires the embeddings from section 7).
metadata = bacpipe.metadata_labels(
    model='birdnet',
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
    overwrite=False,
    return_type='dict',
)
print('Metadata label keys:', list(metadata.keys()))


In [ ]:
# get_dt_filename(): extract the recording time from a bioacoustics filename.
dt = bacpipe.get_dt_filename('CHE_01_20190101_163410.wav')
print(dt)


---
## 9. Model-Specific Evaluation

`model_specific_evaluation` runs the configured evaluation tasks for each
model in a loader dictionary. Here we run **probing** (linear + kNN
classifiers) and **clustering**. The results are saved to disk and reused by
the following sections as well as by the cross-model evaluation.

In [ ]:
bacpipe.model_specific_evaluation(
    loader_dict=loader_dict,
    evaluation_task=['probing', 'clustering'],
    audio_dir='bacpipe/tests/test_data',
    device='cpu',
)
print('Model-specific evaluation completed.')


---
## 10. Probing

The low-level `probing_pipeline` trains (or, with `overwrite=False`, loads) a
linear probe on the embeddings and evaluates it against the ground-truth
labels. `prepare_probe_inference` + `run_probe_inference` then apply a trained
probe to new audio. The results were already computed in section 9, so the
pipeline call simply loads them from disk.

In [ ]:
gt = bacpipe.ground_truth_by_model(
    model='birdnet',
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
    overwrite=False,
)

probe, label2index, metrics = bacpipe.probing_pipeline(
    model_name='birdnet',
    ground_truth=gt,
    embeds=loader_dict['birdnet'].embeddings(return_type='array'),
    paths=get_paths('birdnet'),
    name='linear',
    overwrite=False,
)

print('Probe output classes:', list(label2index.keys()))
print('Overall probing metrics:')
display(metrics)


In [ ]:
# Apply the trained linear probe to the birdnet embeddings and turn the
# predictions into a binary species-presence matrix.
probe, label2index = bacpipe.prepare_probe_inference(
    model='birdnet',
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
    device='cpu',
)

predictions = bacpipe.run_probe_inference(
    model='birdnet',
    linear_probe=probe,
    device='cpu',
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
)

print('Probe predictions shape (n_embeddings, n_classes):', predictions.shape)
display(pd.DataFrame(predictions, columns=list(label2index.keys())).head())


---
## 11. Clustering

Clustering can be used on its own (low-level `run_clustering` +
`eval_clustering` + `eval_with_silhouette`) or through the high-level
`clustering_pipeline`, which runs the algorithms configured in `settings.yaml`
and saves the results.

In [ ]:
from sklearn.cluster import KMeans

embeds = loader_dict['birdnet'].embeddings(return_type='array')
gt_labels = gt['simultaneous_labels'].values

# Low-level: fit a k-means model on the embeddings ...
clusterings = bacpipe.run_clustering(
    embeds=embeds,
    cluster_configs={'kmeans': KMeans(n_clusters=3, n_init=10)},
    ground_truth=gt_labels,
)

# ... and evaluate it against the ground truth and the metadata labels.
clustering_metrics = bacpipe.eval_clustering(
    clusterings,
    ground_truth=gt_labels,
    embeds=embeds,
    metadata_labels=metadata,
)
print('Clustering metrics:')
display(clustering_metrics)

silhouette = bacpipe.eval_with_silhouette(embeds, ground_truth=gt_labels)
print('Silhouette score:')
display(silhouette)


In [ ]:
# High-level: run (or, with overwrite=False, load) the full clustering
# pipeline configured in settings.yaml.
clusterings, clust_results = bacpipe.clustering_pipeline(
    model_name='birdnet',
    ground_truth=gt,
    embeds=embeds,
    paths=get_paths('birdnet'),
    overwrite=False,
)
print('Clustering pipeline results:')
display(clust_results)


---
## 12. Benchmarking

`benchmark` quantifies a model's classifier performance against an annotation
file. Here we compare the plain `birdnet` classifier with a modified `birdnet`
whose batches are standardized before the backbone.

> **Note:** the second call recomputes the `birdnet` embeddings with the
> custom model (which overwrites the plain `birdnet` embeddings on disk), so
> run this section last if you want to keep the plain embeddings.

In [ ]:
import tensorflow as tf
from bacpipe.model_pipelines.feature_extractors.birdnet import Model as BirdnetModel


class NormalizedBirdnet(BirdnetModel):
    # A birdnet variant that standardizes each batch before the backbone.
    def __call__(self, input):
        axes = list(range(1, input.shape.rank))
        mean = tf.reduce_mean(input, axis=axes, keepdims=True)
        std = tf.math.reduce_std(input, axis=axes, keepdims=True)
        return super().__call__((input - mean) / (std + 1e-6))


In [ ]:
# Benchmark the plain birdnet classifier on the test data.
birdnet_report = bacpipe.benchmark(
    model='birdnet',
    dataset='bacpipe/tests/test_data',
    annotations_file='annotations.csv',
    overwrite=False,
)
print('birdnet classifier report:')
display(pd.DataFrame(birdnet_report['report']).T)


In [ ]:
# Benchmark the normalized variant. check_if_already_processed=False forces
# the embeddings/predictions to be recomputed with the custom model.
normalized_report = bacpipe.benchmark(
    model='birdnet',
    dataset='bacpipe/tests/test_data',
    annotations_file='annotations.csv',
    CustomModel=NormalizedBirdnet,
    check_if_already_processed=False,
    overwrite=True,
)
print('normalized birdnet classifier report:')
display(pd.DataFrame(normalized_report['report']).T)


---
## 13. Cross-Model Evaluation

`cross_model_evaluation` generates overview plots that compare the two models
side by side for the probing and clustering tasks, plus a comparison of their
2D embedding spaces.

In [ ]:
bacpipe.cross_model_evaluation(
    audio_dir='bacpipe/tests/test_data',
    models=['birdnet', 'insect459'],
    evaluation_task=['probing', 'clustering'],
    dim_reduction_model='umap',
    device='cpu',
)


In [ ]:
# The overview plots are written to the ``overview`` folder inside the
# results directory. Display all of them at half size.
overview_dir = get_paths('birdnet').plot_path.parent.parent / 'overview'
print('Overview plots in:', overview_dir)
for png in sorted(overview_dir.glob('*.png')):
    print('---', png.name, '---')
    show_half(png)


---
## 14. Dashboard & play

Finally, `visualize_using_dashboard` serves an interactive dashboard for
exploring the embeddings, and `play` runs the entire pipeline in one go. Both
are blocking, so they are shown here as commented-out commands.

In [ ]:
# Run the whole pipeline (embedding generation + evaluation) for the models
# configured in bacpipe.config. This is the highest-level entry point.
#
# ``play`` runs everything in one go and, with ``dashboard=True``, also starts
# the blocking dashboard. It also needs a real ``audio_dir``. Uncomment the
# call below and adapt ``audio_dir`` to run it standalone.

bacpipe.play(
    models=['birdnet', 'insect459'],
    audio_dir='bacpipe/tests/test_data',
    dashboard=False,
)


In [ ]:
# Serve the interactive dashboard for the birdnet embeddings.
#
# ``visualize_using_dashboard`` starts a *blocking* web server (it keeps
# running until you interrupt it with Ctrl-C or click close dashboard).

bacpipe.visualize_using_dashboard(
    models=['birdnet'],
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
)


### Passing your own annotations to the dashboard

The `annotations_df` kwarg is optional. Without it (the cell above) the
dashboard describes every point with the labels `bacpipe` created itself:
metadata labels, ground truth, clusterings and classifier predictions.

If you keep additional information about your recordings in a dataframe -- who
annotated a segment, a confidence value, a site name, ... -- pass it as
`annotations_df`. Every column that is not needed for plotting is then shown
next to the spectrogram of a clicked point (and, with
`only_embed_annotations=True`, the dataframe replaces the annotations csv when
the segments to embed are determined).

The rows are matched to the embedded segments on `audiofilename` and `start`
(the same columns as the annotations csv, `start` in seconds from the beginning
of the file), so the values always belong to the point that was clicked, even
if the dataframe is unsorted or covers only part of the recordings. Segments
without a matching row simply show no value. If `audiofilename` and `start` are
missing, the values are attached by position, which requires exactly one row
per embedding. Anything that cannot be aligned is logged as a warning and the
dashboard is served without the extra columns instead of failing.

In [ ]:
# Serve the dashboard with your own annotations dataframe.
#
# Blocking, just like the cell above.

import pandas as pd

annotations_df = pd.read_csv('bacpipe/tests/test_data/annotations.csv')
# two columns bacpipe knows nothing about
annotations_df['annotator'] = 'reviewer_1'
annotations_df['recording_site'] = [
    'site_a' if 'FewShot' in name else 'site_b'
    for name in annotations_df.audiofilename
]
print(annotations_df.head())

bacpipe.visualize_using_dashboard(
    models=['birdnet'],
    audio_dir='bacpipe/tests/test_data',
    main_results_dir=bacpipe.settings.main_results_dir,
    annotations_df=annotations_df,
)